# Student Hackathon Notebook
## Medical image classification • PyTorch • 2-hour student test

Train a deep neural network, improve it through experiments, and submit predictions to Kaggle.
The instructor's `data_public-3.zip` supplies the original train, validation and test splits. **Do not move images between splits or train on validation/test labels.**

### Your experiment loop
1. Upload data_public-3.zip to Google Drive. Modify your BASE_DIR = Path('/content/drive/MyDrive/teaching/') and run subsequent cells.
2. Edit the model/settings in Step 4, or Step 5
3. Run Step 6: enter what changed and why, then train your model. Each successful experiment adds one row to the CSV and experiment log
4. Run Step 7 to download result csv file and log.
6. Submit your result file to Kaggle: for example: submission_e4f3215394ec4e609b005b363ad1fc63.csv
7. At the end of the Hackathon, submit your simple_experiments.csv and final notebook by email.
8. In your email, please also write
Experiment Reflection:
Write a short paragraph summarising your experiments during the hackathon. Your reflection should address the following:

1). What did you try? Describe the main changes you made to the baseline model.

2). Why did you try them? Explain your reasoning or hypothesis behind these changes.

3). What worked and what did not? Discuss which changes improved performance and which were ineffective, referring to your validation results.

4). What did you learn? Explain what you would do differently if you had more time.


### Setup
Colab normally includes PyTorch and the data-science packages. If imports fail, run the optional install line below, then retry. For a local GPU install, follow [PyTorch's installation instructions](https://pytorch.org/get-started/locally/).

In [26]:
# Optional, only if required packages are missing:
# %pip install torch numpy pandas pillow scikit-learn matplotlib tqdm

from pathlib import Path
import google.colab
IN_COLAB = True

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/teaching/')
ZIP_PATH = BASE_DIR / 'data_public-3.zip'  # EDIT if the ZIP is elsewhere
# Read training images from the fast local Colab disk; keep results on Drive.
EXTRACT_DIR = Path('/content/data')
DATA_DIR = EXTRACT_DIR / 'data_public-3'
OUTPUT_DIR = (Path('/content/drive/MyDrive/hackathon_experiments'))
# You can set OUTPUT_DIR above to another persistent location before training.
print('Data:', DATA_DIR)
print('Results:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data: /content/data/data_public-3
Results: /content/drive/MyDrive/hackathon_experiments


In [27]:
# Step 1. Copy the instructor's ZIP from Drive and extract locally once per runtime.
import shutil
import zipfile

if not DATA_DIR.is_dir():
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f'Dataset ZIP not found: {ZIP_PATH}. Check ZIP_PATH in Setup.')
    local_zip = EXTRACT_DIR.parent / ZIP_PATH.name
    print('Copying dataset ZIP from Drive to local disk...', flush=True)
    shutil.copy2(ZIP_PATH, local_zip)
    print('Extracting dataset locally...', flush=True)
    with zipfile.ZipFile(local_zip) as archive:
        root = EXTRACT_DIR.resolve()
        if any(not (root / item.filename).resolve().is_relative_to(root)
               for item in archive.infolist()):
            raise ValueError('Unsafe path inside ZIP')
        archive.extractall(EXTRACT_DIR)
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f'Expected dataset folder after extraction: {DATA_DIR}')
print('Dataset ready:', DATA_DIR)


Dataset ready: /content/data/data_public-3


In [28]:
# Step 2. Imports
import random
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

print('PyTorch:', torch.__version__, '| GPU available:', torch.cuda.is_available())

PyTorch: 2.11.0+cu130 | GPU available: True


### Step 3. One shared data pipeline
Images are converted to RGB, resized with bilinear interpolation, scaled to `[0, 1]`, and converted to PyTorch's channel-first format. All three splits use this same preprocessing. Add optional augmentation in `augment_image`; it is called only for training images, never for validation or test. Validation and test predictions retain their CSV row order.

In [29]:
# Step 3. Shared preprocessing, with optional training-only augmentation.
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def load_image(path, image_size):
    with Image.open(path) as image:
        image = image.convert('RGB').resize(
            (image_size, image_size), Image.Resampling.BILINEAR
        )
        array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array.copy()).permute(2, 0, 1)  # C, H, W


def augment_image(image):
    # TODO: Add image augmentation here (training images only).
    # image is a float32 tensor with shape [3, height, width], values in [0, 1].
    # Example: randomly flip left/right. Uncomment to try it:
    # if torch.rand(1).item() < 0.5:
    #     image = torch.flip(image, dims=[2])
    return image


class ImageDataset(Dataset):
    def __init__(self, folder, frame, image_size, labeled=True, augment=False):
        self.folder = Path(folder)
        self.frame = frame.reset_index(drop=True).copy()
        self.image_size = image_size
        self.labeled = labeled
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = load_image(self.folder / row['id'], self.image_size)
        if self.augment:
            image = augment_image(image)
        if self.labeled:
            return image, torch.tensor(float(row['label']), dtype=torch.float32)
        return image


def read_split(data_dir, split):
    filename = 'test_ids.csv' if split == 'test' else f'{split}_labels_public.csv'
    frame = pd.read_csv(Path(data_dir) / filename, dtype={'id': str})
    required = {'id'} if split == 'test' else {'id', 'label'}
    if not required.issubset(frame.columns) or frame.empty:
        raise ValueError(f'{filename} must contain {sorted(required)} and at least one row')
    if frame['id'].isna().any() or frame['id'].duplicated().any():
        raise ValueError(f'{filename}: missing or duplicate image IDs')
    if split != 'test' and not frame['label'].isin([0, 1]).all():
        raise ValueError(f'{filename}: labels must be 0 or 1')
    # Training originally used sorted filenames. Validation/test retain CSV order.
    return frame.sort_values('id').reset_index(drop=True) if split == 'train' else frame


def make_loaders(data_dir, config):
    frames = {split: read_split(data_dir, split) for split in ('train', 'val', 'test')}
    loaders = {}
    for split, frame in frames.items():
        dataset = ImageDataset(
            Path(data_dir) / split, frame, config['image_size'],
            labeled=(split != 'test'), augment=(split == 'train'),
        )
        loaders[split] = DataLoader(
            dataset, batch_size=config['batch_size'], shuffle=(split == 'train'),
            num_workers=0, generator=torch.Generator().manual_seed(config['seed']),
        )
    return loaders, frames


def binary_metrics(labels, probabilities, threshold):
    return {
        'auc': float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else None,
        'f1': float(f1_score(labels, probabilities >= threshold, zero_division=0)),
        'accuracy': float(accuracy_score(labels, probabilities >= threshold)),
    }


def run_epoch(model, loader, criterion, device, threshold, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, labels, probabilities = 0.0, [], []
    with torch.set_grad_enabled(training):
        for images, targets in tqdm(loader, desc='Train' if training else 'Validation', unit='batch'):
            images, targets = images.to(device), targets.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images).reshape(-1)

            if logits.shape != targets.shape:
                raise ValueError('Model must return one raw logit per image (no final sigmoid)')
            loss = criterion(logits, targets)
            if not torch.isfinite(loss):
                raise ValueError('Non-finite loss; check model, images and learning rate')
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(targets)
            labels.extend(targets.detach().cpu().numpy())
            probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy())
    metrics = binary_metrics(np.asarray(labels), np.asarray(probabilities), threshold)
    metrics['loss'] = total_loss / len(loader.dataset)
    return metrics


def predict_probabilities(model, loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for images in tqdm(loader, desc='Test predictions', unit='batch'):
            logits = model(images.to(device)).reshape(-1)
            if len(logits) != len(images) or not torch.isfinite(logits).all():
                raise ValueError('Invalid model outputs during test prediction')
            predictions.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(predictions, dtype=np.float64)

### Step 4. Your model and settings
Start with the baseline, then change the code with your ideas. You may replace `build_model` with your own architecture and edit `build_optimizer`. Always return a **new** model. It must output one **raw logit** per image. `BCEWithLogitsLoss` applies the training loss; sigmoid is applied only when calculating probabilities.



In [30]:
# Step 4. STUDENT EDIT AREA: change settings and/or replace build_model.
CONFIG = {
    'image_size': 160,
    'batch_size': 32,
    'epochs': 3,
    'learning_rate': 0.001,
    'weight_decay': 0.0,
    'seed': 42,
    'threshold': 0.5,  # F1/accuracy only; Kaggle receives probabilities
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}


def build_model(config):
    # Simple CNN baseline.
    # Return a NEW model each call. Output one raw logit; do not add Sigmoid.
    size_after_pools = ((config['image_size'] - 2) // 2 - 2) // 2
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(64 * size_after_pools ** 2, 128), nn.ReLU(),
        nn.Linear(128, 1),
    )


def build_optimizer(model, config):
    return torch.optim.Adam(
        model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay']
    )

### Step 5. Simple CSV / Excel logging
Each successful experiment adds one row with just **change, reason, validation AUC, validation F1 and prediction filename**. The last epoch supplies the validation scores and test predictions.

`simple_experiments.csv` keeps the history;  Failed training raises its error without adding a results row.

In [31]:
# One row per successful experiment. CSV is the source for both logs.
from datetime import datetime
from zoneinfo import ZoneInfo

def log_experiment(change, reason, val_auc, val_f1, prediction_file, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / 'simple_experiments.csv'

    row = pd.DataFrame([{
        'timestamp': datetime.now(ZoneInfo("Asia/Singapore")).strftime('%Y-%m-%d %H:%M:%S'),
        'change': change, 'reason': reason,
        'val_auc': val_auc, 'val_f1': val_f1,
        'prediction_file': prediction_file,
    }])
    history = pd.read_csv(csv_path, keep_default_na=False) if csv_path.exists() else pd.DataFrame()
    history = pd.concat([history, row], ignore_index=True)
    history.to_csv(csv_path, index=False)


In [32]:
# STUDENT EDIT AREA: a plain training loop, then predictions and one log call.
def run_experiment(config, change, reason, data_dir, output_dir,
                   model_factory, optimizer_factory):
    if not change.strip() or not reason.strip():
        raise ValueError('Enter the change and reason before training')
    if config['epochs'] < 1:
        raise ValueError('Use at least one training epoch')
    seed_everything(config['seed'])
    device = torch.device(config['device'])

    loaders, frames = make_loaders(data_dir, config)

    model = model_factory(config).to(device)

    optimizer = optimizer_factory(model, config)

    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(1, config['epochs'] + 1):
        print("start training now")
        train = run_epoch(model, loaders['train'], criterion, device,
                          config['threshold'], optimizer)
        validation = run_epoch(model, loaders['val'], criterion, device, config['threshold'])
        print(f"Epoch {epoch}: train loss {train['loss']:.4f}, "
              f"val AUC {validation['auc']}, val F1 {validation['f1']:.4f}")

    predictions = predict_probabilities(model, loaders['test'], device)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_file = f'submission_{uuid.uuid4().hex}.csv'
    pd.DataFrame({'id': frames['test']['id'], 'label': predictions}).to_csv(
        output_dir / prediction_file, index=False, mode='x'
    )
    log_experiment(change, reason, validation['auc'], validation['f1'], prediction_file, output_dir)
    print('Prediction CSV:', output_dir / prediction_file)
    return prediction_file

### Step 6. Run one experiment
Enter a specific change and reason every time. Empty answers are rejected before training starts. The final validation scores and submission use the **last completed epoch**, not an automatically selected best epoch.



In [33]:
CHANGE = input('What did you change? (First run: baseline): ').strip()
REASON = input('Why did you make this change? What do you expect?: ').strip()
run_experiment(
    CONFIG, CHANGE, REASON, DATA_DIR, OUTPUT_DIR,
    model_factory=build_model, optimizer_factory=build_optimizer,
)

What did you change? (First run: baseline): baseline setup to get the first AUC
Why did you make this change? What do you expect?: auc
start training now


Train:   0%|          | 0/36 [00:00<?, ?batch/s]

Validation:   0%|          | 0/6 [00:00<?, ?batch/s]

Epoch 1: train loss 0.5507, val AUC 0.6970636889991728, val F1 0.0000
start training now


Train:   0%|          | 0/36 [00:00<?, ?batch/s]

...

Validation:   0%|          | 0/6 [00:00<?, ?batch/s]

Epoch 3: train loss 0.4833, val AUC 0.7528949545078578, val F1 0.1429


Test predictions:   0%|          | 0/11 [00:00<?, ?batch/s]

Prediction CSV: /content/drive/MyDrive/hackathon_experiments/submission_9ad7d7fdda464a1380d942df58bf6280.csv


'submission_9ad7d7fdda464a1380d942df58bf6280.csv'

In [34]:
# Review all successful experiments, including their prediction filenames.
display(pd.read_csv(OUTPUT_DIR / 'simple_experiments.csv'))

,timestamp,change,reason,val_auc,val_f1,prediction_file
0,2026-10-12 22:32:00,baseline setup to get the first AUC,auc,0.752895,0.142857,submission_9ad7d7fdda464a1380d942df58bf6280.csv


### Step 7. Submit and keep your results
Upload your chosen prediction CSV to Kaggle yourself. Its filename is recorded beside the change, reason and validation scores. Download the results folder below and save this edited notebook.

In [ ]:
# Optional: rebuild Excel from CSV after fixing an Excel export warning.
# pd.read_csv(OUTPUT_DIR / 'simple_experiments.csv', keep_default_na=False).to_excel(
#     OUTPUT_DIR / 'simple_experiments.xlsx', index=False, engine='openpyxl'
# )

In [35]:
# Download the results folder, including all run-linked submissions and logs.
# Run this before leaving Colab if OUTPUT_DIR is not on Drive.
#import shutil

archive_base = OUTPUT_DIR.parent / ('hackathon_results_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR)
print('Results archive:', archive_path)
if IN_COLAB:
    from google.colab import files
    files.download(archive_path)
# Also save/download this edited .ipynb notebook from the notebook's File menu.

Results archive: /content/drive/MyDrive/hackathon_results_20261012T143212.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>